# 0. 환경설정

In [6]:
import os
import sys
import platform
import warnings

# [1] Protobuf 에러 및 로그 레벨 설정 (최상단 고정)
os.environ['PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION'] = 'python'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2' # 정보 로그까지는 확인 (오류 추적용)
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

import tensorflow as tf
from tensorflow.keras import mixed_precision
import torch
import numpy as np

# 0. 경고 메시지 억제
warnings.filterwarnings("ignore")

print("=" * 70)
print("     📸 이미지 분류(CNN) 학습 최적화 환경 설정 (VRAM 4GB)")
print("=" * 70)

# 1. TensorFlow CNN 학습 최적화 (메모리 관리)
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        # 동적 할당: CNN 모델 크기에 따라 필요한 만큼만 점유
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        
        # 4GB VRAM 고려: 이미지 데이터가 많을 경우를 위해 한도 상향 (2.0GB)
        # 텍스트보다 이미지가 메모리를 더 많이 먹으므로 1.5 -> 2.0으로 조정 제안
        tf.config.set_logical_device_configuration(
            gpus[0],
            [tf.config.LogicalDeviceConfiguration(memory_limit=2048)]
        )
        
        # [CNN 핵심] 혼합 정밀도: 학습 속도를 높이고 VRAM 사용량을 획기적으로 줄임
        policy = mixed_precision.Policy('mixed_float16')
        mixed_precision.set_global_policy(policy)
        
        print(f"✅ TF GPU: CNN 가속 모드 활성화 (Limit: 2GB / Mixed Precision)")
    except Exception as e:
        print(f"ℹ️ GPU 설정 정보: {e}")

# 2. PyTorch 및 기타 디바이스 확인
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"\n[디바이스 가용 상태]")
print(f"TensorFlow Version: {tf.__version__}")
print(f"CNN Acceleration: {'AVAILABLE (GPU)' if gpus else 'CPU ONLY'}")
print(f"이미지 정규화 정책: float16 (Mixed Precision)")

# 3. 이미지 데이터 규격 가이드 (박사님의 계획 반영)
print("\n" + "=" * 70)
print(f"💡 CNN 모델 구성 팁 (STEP 4 전용):")
print(f" - Input Shape: (64, 128, 1)  # 64x128 흑백 이미지")
print(f" - Batch Size: 16 또는 32 권장 (4GB VRAM 고려)")
print("=" * 70 + "\n")

     📸 이미지 분류(CNN) 학습 최적화 환경 설정 (VRAM 4GB)
ℹ️ GPU 설정 정보: Cannot set memory growth on device when virtual devices configured

[디바이스 가용 상태]
TensorFlow Version: 2.10.1
CNN Acceleration: AVAILABLE (GPU)
이미지 정규화 정책: float16 (Mixed Precision)

💡 CNN 모델 구성 팁 (STEP 4 전용):
 - Input Shape: (64, 128, 1)  # 64x128 흑백 이미지
 - Batch Size: 16 또는 32 권장 (4GB VRAM 고려)



#1. 데이터 생성

In [12]:
import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont, ImageFilter, ImageEnhance
import glob
from datetime import datetime
import json
import cv2

# 1. 생성일시를 포함한 데이터 폴더 생성
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
DATA_DIR = f"robot_camera_dataset_{timestamp}"
CATEGORIES = ["go", "stop"]  # 0: 앞으로 가, 1: 멈춰

for cat in CATEGORIES:
    os.makedirs(os.path.join(DATA_DIR, cat), exist_ok=True)

print(f"📁 데이터셋 폴더 생성: {DATA_DIR}")
print(f"🤖 PiCar-X AI Car 카메라 시뮬레이션 모드")

# 2. 폰트 설정 (기존과 동일)
NORMAL_FONTS = [
    "C:/Windows/Fonts/malgun.ttf",
    "C:/Windows/Fonts/malgunbd.ttf",
    "C:/Windows/Fonts/batang.ttc",
    "C:/Windows/Fonts/batangche.ttc",
    "C:/Windows/Fonts/gulim.ttc",
    "C:/Windows/Fonts/gulimche.ttc",
    "C:/Windows/Fonts/gungsuh.ttc",
    "C:/Windows/Fonts/dotum.ttc",
    "C:/Windows/Fonts/dotumche.ttc",
    "C:/Windows/Fonts/NanumGothic.ttf",
    "C:/Windows/Fonts/NanumMyeongjo.ttf",
]

DISTORTED_FONTS = [
    "C:/Windows/Fonts/NanumBrush.ttf",
    "C:/Windows/Fonts/NanumPen.ttf",
    "C:/Windows/Fonts/HYKANM.TTF",
    "C:/Windows/Fonts/HYGTRE.TTF",
    "C:/Windows/Fonts/HYPOST.TTF",
    "C:/Windows/Fonts/BMEULJI.TTF",
    "C:/Windows/Fonts/BMJUA.TTF",
    "C:/Windows/Fonts/BMHANNA.TTF",
]

def get_available_fonts():
    """폰트를 일반/변형으로 분류하여 반환"""
    normal_available = []
    distorted_available = []
    
    for font_path in NORMAL_FONTS:
        if os.path.exists(font_path):
            normal_available.append(font_path)
    
    for font_path in DISTORTED_FONTS:
        if os.path.exists(font_path):
            distorted_available.append(font_path)
    
    if not normal_available:
        normal_available = ["C:/Windows/Fonts/malgun.ttf"]
    
    if not distorted_available:
        distorted_available = normal_available.copy()
    
    print(f"\n✅ 일반 폰트: {len(normal_available)}개")
    print(f"✅ 변형 폰트: {len(distorted_available)}개")
    
    return normal_available, distorted_available

# 3. 카메라 환경 파라미터 정의
class CameraEnvironment:
    """PiCar-X AI Car 카메라 환경 시뮬레이션"""
    
    # 거리 파라미터 (cm 단위)
    DISTANCE_PARAMS = {
        'very_close': {'range': (10, 20), 'blur': (0, 1), 'scale': (1.8, 2.5), 'distortion': 0.03},
        'close': {'range': (20, 40), 'blur': (0, 0.5), 'scale': (1.3, 1.8), 'distortion': 0.02},
        'optimal': {'range': (40, 80), 'blur': (0, 0.2), 'scale': (1.0, 1.3), 'distortion': 0.01},
        'far': {'range': (80, 120), 'blur': (0.5, 1.5), 'scale': (0.6, 1.0), 'distortion': 0.02},
        'very_far': {'range': (120, 200), 'blur': (1.5, 3.0), 'scale': (0.3, 0.6), 'distortion': 0.03},
    }
    
    # 조도 파라미터 (lux 단위)
    ILLUMINATION_PARAMS = {
        'very_dark': {'range': (1, 50), 'brightness': (0.3, 0.5), 'noise_level': 0.25, 'contrast': (0.6, 0.8)},
        'dark': {'range': (50, 150), 'brightness': (0.5, 0.7), 'noise_level': 0.15, 'contrast': (0.7, 0.9)},
        'normal': {'range': (150, 500), 'brightness': (0.8, 1.2), 'noise_level': 0.05, 'contrast': (0.9, 1.1)},
        'bright': {'range': (500, 1000), 'brightness': (1.2, 1.5), 'noise_level': 0.08, 'contrast': (1.1, 1.3)},
        'very_bright': {'range': (1000, 2000), 'brightness': (1.5, 2.0), 'noise_level': 0.12, 'contrast': (0.8, 1.0)},
    }
    
    # 배경 타입
    BACKGROUND_TYPES = {
        'plain_white': {'texture_noise': 0.02, 'pattern_intensity': 0},
        'plain_gray': {'texture_noise': 0.05, 'pattern_intensity': 0},
        'textured': {'texture_noise': 0.15, 'pattern_intensity': 0.3},
        'complex': {'texture_noise': 0.25, 'pattern_intensity': 0.5},
        'outdoor': {'texture_noise': 0.20, 'pattern_intensity': 0.4},
    }
    
    @staticmethod
    def sample_distance():
        """거리 조건 샘플링"""
        dist_type = np.random.choice(list(CameraEnvironment.DISTANCE_PARAMS.keys()))
        params = CameraEnvironment.DISTANCE_PARAMS[dist_type]
        distance_cm = np.random.uniform(*params['range'])
        return dist_type, distance_cm, params
    
    @staticmethod
    def sample_illumination():
        """조도 조건 샘플링"""
        illum_type = np.random.choice(list(CameraEnvironment.ILLUMINATION_PARAMS.keys()))
        params = CameraEnvironment.ILLUMINATION_PARAMS[illum_type]
        lux = np.random.uniform(*params['range'])
        return illum_type, lux, params
    
    @staticmethod
    def sample_background():
        """배경 타입 샘플링"""
        bg_type = np.random.choice(list(CameraEnvironment.BACKGROUND_TYPES.keys()))
        params = CameraEnvironment.BACKGROUND_TYPES[bg_type]
        return bg_type, params

def generate_background(width, height, bg_type, bg_params):
    """배경 생성"""
    # 기본 배경 색상
    if bg_type == 'plain_white':
        base_color = np.random.randint(240, 256)
    elif bg_type == 'plain_gray':
        base_color = np.random.randint(180, 220)
    elif bg_type == 'outdoor':
        base_color = np.random.randint(100, 180)
    else:
        base_color = np.random.randint(150, 230)
    
    background = np.ones((height, width), dtype=np.uint8) * base_color
    
    # 텍스처 노이즈 추가
    texture_noise = np.random.normal(0, bg_params['texture_noise'] * 255, (height, width))
    background = np.clip(background + texture_noise, 0, 255).astype(np.uint8)
    
    # 패턴 추가 (복잡한 배경)
    if bg_params['pattern_intensity'] > 0:
        # 랜덤 패턴 (선, 점 등)
        num_patterns = int(bg_params['pattern_intensity'] * 20)
        for _ in range(num_patterns):
            if np.random.rand() < 0.5:
                # 랜덤 선
                x1, y1 = np.random.randint(0, width), np.random.randint(0, height)
                x2, y2 = np.random.randint(0, width), np.random.randint(0, height)
                color = np.random.randint(50, 200)
                cv2.line(background, (x1, y1), (x2, y2), int(color), 1)
            else:
                # 랜덤 원
                cx, cy = np.random.randint(0, width), np.random.randint(0, height)
                radius = np.random.randint(5, 30)
                color = np.random.randint(50, 200)
                cv2.circle(background, (cx, cy), radius, int(color), -1)
    
    return background

def apply_camera_effects(img, dist_params, illum_params, bg_type, bg_params):
    """카메라 효과 적용"""
    img_array = np.array(img, dtype=np.float32)
    
    # 1. 거리에 따른 효과
    # 스케일 변화
    scale = np.random.uniform(*dist_params['scale'])
    new_size = (int(img.width * scale), int(img.height * scale))
    img_scaled = img.resize(new_size, Image.LANCZOS)
    
    # 중앙에 배치 (배경과 합성)
    background = generate_background(128, 64, bg_type, bg_params)
    img_bg = Image.fromarray(background)
    
    x_offset = (128 - new_size[0]) // 2
    y_offset = (64 - new_size[1]) // 2
    
    # 스케일된 이미지를 배경에 붙이기
    if x_offset >= 0 and y_offset >= 0:
        img_bg.paste(img_scaled, (x_offset, y_offset))
    else:
        # 이미지가 너무 크면 크롭
        crop_x = max(0, -x_offset)
        crop_y = max(0, -y_offset)
        img_scaled_cropped = img_scaled.crop((crop_x, crop_y, crop_x + 128, crop_y + 64))
        img_bg.paste(img_scaled_cropped, (max(0, x_offset), max(0, y_offset)))
    
    img_array = np.array(img_bg, dtype=np.float32)
    
    # 블러 (거리에 따른 초점 흐림)
    blur_amount = np.random.uniform(*dist_params['blur'])
    if blur_amount > 0:
        blur_kernel = int(blur_amount * 2) * 2 + 1
        img_array = cv2.GaussianBlur(img_array, (blur_kernel, blur_kernel), blur_amount)
    
    # 왜곡 (렌즈 왜곡)
    if dist_params['distortion'] > 0:
        distortion = np.random.uniform(0, dist_params['distortion'])
        noise = np.random.normal(0, distortion * 10, img_array.shape)
        img_array = np.clip(img_array + noise, 0, 255)
    
    # 2. 조도에 따른 효과
    # 밝기 조정
    brightness = np.random.uniform(*illum_params['brightness'])
    img_array = img_array * brightness
    
    # 대비 조정
    contrast = np.random.uniform(*illum_params['contrast'])
    img_array = (img_array - 128) * contrast + 128
    
    # 노이즈 (조도 노이즈)
    noise_level = illum_params['noise_level']
    noise = np.random.normal(0, noise_level * 255, img_array.shape)
    img_array = img_array + noise
    
    # 클리핑
    img_array = np.clip(img_array, 0, 255).astype(np.uint8)
    
    # 3. 카메라 센서 노이즈 (전자 노이즈)
    sensor_noise = np.random.normal(0, 3, img_array.shape)
    img_array = np.clip(img_array + sensor_noise, 0, 255).astype(np.uint8)
    
    return Image.fromarray(img_array)

def create_base_text_image(text, font_path, font_size):
    """기본 텍스트 이미지 생성 (배경 투명)"""
    # 큰 캔버스에 먼저 그리기
    temp_img = Image.new('L', (256, 128), color=0)
    draw = ImageDraw.Draw(temp_img)
    
    try:
        font = ImageFont.truetype(font_path, font_size)
        font_loaded = True
    except:
        font = ImageFont.load_default()
        font_loaded = False
    
    # 텍스트 크기 계산
    try:
        bbox = draw.textbbox((0, 0), text, font=font)
        w, h = bbox[2] - bbox[0], bbox[3] - bbox[1]
    except:
        w, h = draw.textsize(text, font=font)
        bbox = (0, 0, w, h)
    
    # 중앙 배치
    x = (256 - w) // 2
    y = (128 - h) // 2 - bbox[1]
    
    # 텍스트 그리기 (흰색)
    draw.text((x, y), text, font=font, fill=255)
    
    # 128x64로 리사이즈
    img = temp_img.resize((128, 64), Image.LANCZOS)
    
    return img, font_loaded, os.path.basename(font_path) if font_loaded else 'default'

def save_camera_images(text, category, count, fonts, font_type_name):
    """
    카메라 환경을 시뮬레이션한 이미지 생성
    
    Args:
        text: 생성할 텍스트
        category: 카테고리 (go/stop)
        count: 생성할 이미지 수
        fonts: 사용할 폰트 리스트
        font_type_name: 폰트 타입 이름 ('normal' or 'distorted')
    
    Returns:
        metadata: 각 이미지의 메타데이터
    """
    print(f"\n📸 '{text}' 카메라 이미지 생성 중 ({category}, {font_type_name})...")
    print(f"   생성 수: {count}개")
    
    metadata = []
    
    for i in range(count):
        # 1. 폰트 선택
        f_path = np.random.choice(fonts)
        f_size = np.random.randint(20, 40)
        
        # 2. 기본 텍스트 이미지 생성
        base_img, font_loaded, font_name = create_base_text_image(text, f_path, f_size)
        
        # 3. 카메라 환경 파라미터 샘플링
        dist_type, distance_cm, dist_params = CameraEnvironment.sample_distance()
        illum_type, lux, illum_params = CameraEnvironment.sample_illumination()
        bg_type, bg_params = CameraEnvironment.sample_background()
        
        # 4. 카메라 효과 적용
        final_img = apply_camera_effects(base_img, dist_params, illum_params, bg_type, bg_params)
        
        # 5. 저장
        file_name = f"{category}_{i:04d}_{font_type_name}.png"
        file_path = os.path.join(DATA_DIR, category, file_name)
        final_img.save(file_path)
        
        # 6. 메타데이터 저장
        metadata.append({
            'file_name': file_name,
            'category': category,
            'font_type': font_type_name,
            'font_path': font_name,
            'font_size': f_size,
            
            # 카메라 환경 파라미터
            'camera_params': {
                'distance': {
                    'type': dist_type,
                    'value_cm': round(distance_cm, 2),
                    'blur_range': dist_params['blur'],
                    'scale_range': dist_params['scale'],
                    'distortion': dist_params['distortion']
                },
                'illumination': {
                    'type': illum_type,
                    'value_lux': round(lux, 2),
                    'brightness_range': illum_params['brightness'],
                    'noise_level': illum_params['noise_level'],
                    'contrast_range': illum_params['contrast']
                },
                'background': {
                    'type': bg_type,
                    'texture_noise': bg_params['texture_noise'],
                    'pattern_intensity': bg_params['pattern_intensity']
                }
            }
        })
        
        if (i + 1) % 100 == 0:
            print(f"  진행: {i + 1}/{count}")
    
    print(f"✅ {count}개 이미지 저장 완료 ({font_type_name})")
    
    return metadata

# --- 실행: 카메라 데이터셋 생성 (총 2000개) ---
normal_fonts, distorted_fonts = get_available_fonts()

print("\n" + "="*60)
print("🚀 PiCar-X AI Car 카메라 데이터셋 생성 시작")
print("="*60)
print("📊 구성:")
print("   - 일반 폰트: 1000개 (500개씩)")
print("   - 변형 폰트: 1000개 (500개씩)")
print("   - 총 2000개 이미지")
print("\n🎥 카메라 환경 파라미터:")
print("   - 거리: 10cm ~ 200cm (5단계)")
print("   - 조도: 1 ~ 2000 lux (5단계)")
print("   - 배경: 5가지 타입")
print("="*60)

all_metadata = []

# "앞으로 가" - 일반 폰트
metadata_go_normal = save_camera_images(
    text="앞으로 가",
    category="go",
    count=500,
    fonts=normal_fonts,
    font_type_name="normal"
)
all_metadata.extend(metadata_go_normal)

# "앞으로 가" - 변형 폰트
metadata_go_distorted = save_camera_images(
    text="앞으로 가",
    category="go",
    count=500,
    fonts=distorted_fonts,
    font_type_name="distorted"
)
all_metadata.extend(metadata_go_distorted)

# "멈춰" - 일반 폰트
metadata_stop_normal = save_camera_images(
    text="멈춰",
    category="stop",
    count=500,
    fonts=normal_fonts,
    font_type_name="normal"
)
all_metadata.extend(metadata_stop_normal)

# "멈춰" - 변형 폰트
metadata_stop_distorted = save_camera_images(
    text="멈춰",
    category="stop",
    count=500,
    fonts=distorted_fonts,
    font_type_name="distorted"
)
all_metadata.extend(metadata_stop_distorted)

# --- 메타데이터 저장 (JSON) ---
metadata_path = os.path.join(DATA_DIR, "camera_dataset_metadata.json")
with open(metadata_path, 'w', encoding='utf-8') as f:
    json.dump(all_metadata, f, ensure_ascii=False, indent=2)

print(f"\n📝 메타데이터 저장: {metadata_path}")

# --- 통계 정보 생성 ---
print("\n" + "="*60)
print("✅ 카메라 데이터셋 구축 완료")
print("="*60)
print(f"📁 저장 위치: {DATA_DIR}")
print(f"📊 총 이미지 수: 2000개")
print(f"\n📈 카테고리별 분포:")
print(f"   - 'go' (앞으로 가): 1000개")
print(f"      • 일반 폰트: 500개")
print(f"      • 변형 폰트: 500개")
print(f"   - 'stop' (멈춰): 1000개")
print(f"      • 일반 폰트: 500개")
print(f"      • 변형 폰트: 500개")

# --- 환경 파라미터 통계 ---
distance_stats = {}
illumination_stats = {}
background_stats = {}

for m in all_metadata:
    # 거리 통계
    dist_type = m['camera_params']['distance']['type']
    distance_stats[dist_type] = distance_stats.get(dist_type, 0) + 1
    
    # 조도 통계
    illum_type = m['camera_params']['illumination']['type']
    illumination_stats[illum_type] = illumination_stats.get(illum_type, 0) + 1
    
    # 배경 통계
    bg_type = m['camera_params']['background']['type']
    background_stats[bg_type] = background_stats.get(bg_type, 0) + 1

print(f"\n🎥 카메라 환경 파라미터 분포:")
print(f"\n📏 거리 분포:")
for dist_type, count in sorted(distance_stats.items()):
    print(f"   - {dist_type}: {count}개 ({count/20:.1f}%)")

print(f"\n💡 조도 분포:")
for illum_type, count in sorted(illumination_stats.items()):
    print(f"   - {illum_type}: {count}개 ({count/20:.1f}%)")

print(f"\n🖼️  배경 분포:")
for bg_type, count in sorted(background_stats.items()):
    print(f"   - {bg_type}: {count}개 ({count/20:.1f}%)")

# --- 샘플 이미지 시각화 ---
def visualize_camera_samples():
    """카메라 환경별 샘플 시각화"""
    fig = plt.figure(figsize=(20, 12))
    
    categories_display = ["앞으로 가", "멈춰"]
    font_types = ['normal', 'distorted']
    
    sample_conditions = [
        ('very_close', 'very_dark', 'plain_white'),
        ('optimal', 'normal', 'plain_gray'),
        ('far', 'bright', 'textured'),
        ('very_far', 'very_bright', 'complex'),
    ]
    
    plot_idx = 1
    
    for cat_idx, cat in enumerate(CATEGORIES):
        for font_idx, font_type in enumerate(font_types):
            # 해당 조건의 이미지 찾기
            for cond_idx, (dist, illum, bg) in enumerate(sample_conditions):
                # 조건에 맞는 이미지 찾기
                matching_images = [
                    m for m in all_metadata
                    if m['category'] == cat
                    and m['font_type'] == font_type
                    and m['camera_params']['distance']['type'] == dist
                    and m['camera_params']['illumination']['type'] == illum
                    and m['camera_params']['background']['type'] == bg
                ]
                
                if matching_images:
                    m = matching_images[0]
                    img_path = os.path.join(DATA_DIR, cat, m['file_name'])
                    img = Image.open(img_path)
                    
                    ax = plt.subplot(4, 8, plot_idx)
                    ax.imshow(img, cmap='gray')
                    ax.axis('off')
                    
                    # 제목
                    title = f"{categories_display[cat_idx]}\n{font_type}"
                    if cond_idx == 0:
                        ax.set_ylabel(title, fontsize=10, rotation=0, ha='right', va='center')
                    
                    # 조건 표시
                    condition_text = f"{dist}\n{illum}\n{bg}"
                    ax.set_title(condition_text, fontsize=7)
                    
                    plot_idx += 1
    
    plt.suptitle('PiCar-X 카메라 환경별 샘플 이미지', fontsize=16, fontweight='bold')
    plt.tight_layout()
    
    viz_path = os.path.join(DATA_DIR, "camera_samples.png")
    plt.savefig(viz_path, dpi=150, bbox_inches='tight')
    print(f"\n📊 카메라 샘플 이미지 저장: {viz_path}")
    plt.close()

visualize_camera_samples()

# --- 요약 정보 저장 (텍스트) ---
summary_path = os.path.join(DATA_DIR, "dataset_info.txt")
with open(summary_path, 'w', encoding='utf-8') as f:
    f.write(f"PiCar-X AI Car 카메라 데이터셋 정보\n")
    f.write(f"="*60 + "\n")
    f.write(f"생성 일시: {timestamp}\n\n")
    
    f.write(f"📊 전체 통계\n")
    f.write(f"-"*60 + "\n")
    f.write(f"총 이미지 수: 2000개\n")
    f.write(f"  - 일반 폰트: 1000개 (50%)\n")
    f.write(f"  - 변형 폰트: 1000개 (50%)\n\n")
    
    f.write(f"📈 카테고리별 분포\n")
    f.write(f"-"*60 + "\n")
    f.write(f"'go' (앞으로 가): 1000개\n")
    f.write(f"  - 일반 폰트: 500개\n")
    f.write(f"  - 변형 폰트: 500개\n\n")
    f.write(f"'stop' (멈춰): 1000개\n")
    f.write(f"  - 일반 폰트: 500개\n")
    f.write(f"  - 변형 폰트: 500개\n\n")
    
    f.write(f"🎥 카메라 환경 파라미터\n")
    f.write(f"-"*60 + "\n")
    f.write(f"\n📏 거리 조건 (5단계):\n")
    for dist_type, params in CameraEnvironment.DISTANCE_PARAMS.items():
        f.write(f"  - {dist_type}: {params['range'][0]}~{params['range'][1]}cm\n")
        f.write(f"    블러: {params['blur']}, 스케일: {params['scale']}\n")
    
    f.write(f"\n💡 조도 조건 (5단계):\n")
    for illum_type, params in CameraEnvironment.ILLUMINATION_PARAMS.items():
        f.write(f"  - {illum_type}: {params['range'][0]}~{params['range'][1]} lux\n")
        f.write(f"    밝기: {params['brightness']}, 노이즈: {params['noise_level']}\n")
    
    f.write(f"\n🖼️  배경 조건 (5가지):\n")
    for bg_type, params in CameraEnvironment.BACKGROUND_TYPES.items():
        f.write(f"  - {bg_type}: 텍스처={params['texture_noise']}, 패턴={params['pattern_intensity']}\n")
    
    f.write(f"\n📊 환경 파라미터 분포:\n")
    f.write(f"-"*60 + "\n")
    f.write(f"\n거리 분포:\n")
    for dist_type, count in sorted(distance_stats.items()):
        f.write(f"  - {dist_type}: {count}개 ({count/20:.1f}%)\n")
    
    f.write(f"\n조도 분포:\n")
    for illum_type, count in sorted(illumination_stats.items()):
        f.write(f"  - {illum_type}: {count}개 ({count/20:.1f}%)\n")
    
    f.write(f"\n배경 분포:\n")
    for bg_type, count in sorted(background_stats.items()):
        f.write(f"  - {bg_type}: {count}개 ({count/20:.1f}%)\n")

print(f"📝 요약 정보 저장: {summary_path}")

print("\n" + "="*60)
print("✨ 모든 작업 완료!")
print("="*60)
print(f"\n📦 생성된 파일:")
print(f"   1. 이미지 파일: {DATA_DIR}/go/ 및 {DATA_DIR}/stop/")
print(f"   2. 메타데이터: {metadata_path}")
print(f"   3. 요약 정보: {summary_path}")
print(f"   4. 샘플 시각화: {os.path.join(DATA_DIR, 'camera_samples.png')}")
print(f"\n🎥 카메라 환경 파라미터가 메타데이터에 기록되었습니다.")
print(f"💡 각 이미지의 거리, 조도, 배경 정보는 JSON 파일에서 확인하세요.")

📁 데이터셋 폴더 생성: robot_camera_dataset_20260218_212132
🤖 PiCar-X AI Car 카메라 시뮬레이션 모드

✅ 일반 폰트: 4개
✅ 변형 폰트: 2개

🚀 PiCar-X AI Car 카메라 데이터셋 생성 시작
📊 구성:
   - 일반 폰트: 1000개 (500개씩)
   - 변형 폰트: 1000개 (500개씩)
   - 총 2000개 이미지

🎥 카메라 환경 파라미터:
   - 거리: 10cm ~ 200cm (5단계)
   - 조도: 1 ~ 2000 lux (5단계)
   - 배경: 5가지 타입

📸 '앞으로 가' 카메라 이미지 생성 중 (go, normal)...
   생성 수: 500개
  진행: 100/500
  진행: 200/500
  진행: 300/500
  진행: 400/500
  진행: 500/500
✅ 500개 이미지 저장 완료 (normal)

📸 '앞으로 가' 카메라 이미지 생성 중 (go, distorted)...
   생성 수: 500개
  진행: 100/500
  진행: 200/500
  진행: 300/500
  진행: 400/500
  진행: 500/500
✅ 500개 이미지 저장 완료 (distorted)

📸 '멈춰' 카메라 이미지 생성 중 (stop, normal)...
   생성 수: 500개
  진행: 100/500
  진행: 200/500
  진행: 300/500
  진행: 400/500
  진행: 500/500
✅ 500개 이미지 저장 완료 (normal)

📸 '멈춰' 카메라 이미지 생성 중 (stop, distorted)...
   생성 수: 500개
  진행: 100/500
  진행: 200/500
  진행: 300/500
  진행: 400/500
  진행: 500/500
✅ 500개 이미지 저장 완료 (distorted)

📝 메타데이터 저장: robot_camera_dataset_20260218_212132\camera_dataset_metadata.json

✅ 카메라 데이

#2. 분류모델

In [10]:
import os
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import seaborn as sns
from PIL import Image
import glob
from datetime import datetime
import json
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from sklearn.metrics import precision_recall_fscore_support, roc_curve, auc
import pandas as pd
from scipy.interpolate import griddata
import pickle

# 한글 폰트 설정
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

print("="*60)
print("🤖 PiCar-X 로봇 카메라 이미지 분류 및 노이즈 영향도 분석")
print("="*60)

# ==================== 1. 데이터 폴더 선택 ====================
def select_dataset_folder_manual():
    """수동으로 데이터셋 폴더 경로 입력"""
    print("\n📁 데이터셋 폴더 경로를 입력해주세요.")
    print()
    
    current_dir = os.getcwd()
    robot_folders = [f for f in os.listdir(current_dir) if f.startswith('robot_camera_dataset_')]
    
    if robot_folders:
        print("🔍 현재 디렉토리에서 발견된 데이터셋 폴더:")
        for idx, folder in enumerate(robot_folders, 1):
            print(f"   {idx}. {folder}")
        print()
        
        choice = input("📌 번호를 입력하거나 전체 경로를 입력하세요: ").strip()
        
        if choice.isdigit():
            idx = int(choice) - 1
            if 0 <= idx < len(robot_folders):
                folder_path = os.path.join(current_dir, robot_folders[idx])
            else:
                print("❌ 잘못된 번호입니다.")
                return None, None
        else:
            if os.path.isabs(choice):
                folder_path = choice
            else:
                folder_path = os.path.join(current_dir, choice)
    else:
        print("⚠️  현재 디렉토리에 데이터셋 폴더가 없습니다.")
        folder_path = input("📌 전체 경로를 입력하세요: ").strip()
    
    folder_path = folder_path.strip('"').strip("'")
    
    if not os.path.exists(folder_path):
        print(f"❌ 폴더를 찾을 수 없습니다: {folder_path}")
        return None, None
    
    metadata_path = os.path.join(folder_path, "camera_dataset_metadata.json")
    if not os.path.exists(metadata_path):
        print(f"❌ 메타데이터 파일을 찾을 수 없습니다")
        return None, None
    
    print(f"\n✅ 선택된 폴더: {folder_path}")
    
    return folder_path, metadata_path

DATA_DIR, metadata_path = select_dataset_folder_manual()

if DATA_DIR is None:
    print("\n❌ 프로그램을 종료합니다.")
    exit()

RESULTS_DIR = os.path.join(DATA_DIR, "analysis_results")
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f"📊 결과 저장: {RESULTS_DIR}")

# ==================== 2. 메타데이터 로드 ====================
print("\n📂 메타데이터 로딩...")
with open(metadata_path, 'r', encoding='utf-8') as f:
    all_metadata = json.load(f)

print(f"✅ {len(all_metadata)}개 로드")

# 메타데이터 수정
for m in all_metadata:
    if 'label' not in m:
        m['label'] = 0 if m['category'] == 'go' else 1
    
    if 'score' not in m['camera_params']['distance']:
        score_map = {'very_close': 1, 'close': 2, 'optimal': 3, 'far': 4, 'very_far': 5}
        m['camera_params']['distance']['score'] = score_map.get(m['camera_params']['distance']['type'], 3)
    
    if 'score' not in m['camera_params']['illumination']:
        score_map = {'very_dark': 1, 'dark': 2, 'normal': 3, 'bright': 4, 'very_bright': 5}
        m['camera_params']['illumination']['score'] = score_map.get(m['camera_params']['illumination']['type'], 3)
    
    if 'score' not in m['camera_params']['background']:
        score_map = {'plain_white': 1, 'plain_gray': 2, 'textured': 3, 'outdoor': 4, 'complex': 5}
        m['camera_params']['background']['score'] = score_map.get(m['camera_params']['background']['type'], 3)

# ==================== 3. 데이터셋 로드 ====================
print("\n🖼️  이미지 데이터 로딩...")

def load_dataset(metadata, data_dir):
    images = []
    labels = []
    valid_metadata = []
    
    failed_count = 0
    for i, m in enumerate(metadata):
        img_path = os.path.join(data_dir, m['category'], m['file_name'])
        
        if not os.path.exists(img_path):
            failed_count += 1
            continue
        
        try:
            img = Image.open(img_path).convert('L')
            img_array = np.array(img).flatten() / 255.0  # Flatten for sklearn
            images.append(img_array)
            labels.append(m['label'])
            valid_metadata.append(m)
        except:
            failed_count += 1
            continue
        
        if (i + 1) % 500 == 0:
            print(f"  진행: {i + 1}/{len(metadata)}")
    
    return np.array(images), np.array(labels), valid_metadata

X, y, valid_metadata = load_dataset(all_metadata, DATA_DIR)

if len(X) == 0:
    print("\n❌ 로드된 이미지가 없습니다.")
    exit()

print(f"\n✅ 총 {len(X)}개 이미지 로드")
print(f"  - shape: {X.shape}")
print(f"  - go={np.sum(y==0)}개, stop={np.sum(y==1)}개")

X_train, X_test, y_train, y_test, meta_train, meta_test = train_test_split(
    X, y, valid_metadata, test_size=0.2, random_state=42, stratify=y
)

print(f"\n📊 데이터 분할:")
print(f"  - 학습: {X_train.shape[0]}개")
print(f"  - 테스트: {X_test.shape[0]}개")

# ==================== 4. 모델 학습 ====================
print("\n" + "="*60)
print("🧠 신경망 모델 학습")
print("="*60)

clf = MLPClassifier(
    hidden_layer_sizes=(256, 128, 64),
    max_iter=1000,
    random_state=42,
    verbose=False,
    early_stopping=True,
    validation_fraction=0.1,
    learning_rate_init=0.001
)

start_time = datetime.now()
clf.fit(X_train, y_train)
end_time = datetime.now()
training_time = (end_time - start_time).total_seconds()

print(f"✅ 학습 완료 (소요: {training_time:.2f}초)")
print(f"📊 반복: {clf.n_iter_}회")

# 모델 저장
model_path = os.path.join(RESULTS_DIR, "robot_classifier_model.pkl")
with open(model_path, 'wb') as f:
    pickle.dump(clf, f)
print(f"💾 모델 저장: {model_path}")

# ==================== 5. 모델 평가 ====================
print("\n" + "="*60)
print("📈 모델 평가")
print("="*60)

y_pred = clf.predict(X_test)
y_pred_proba = clf.predict_proba(X_test)

test_acc = accuracy_score(y_test, y_pred)
print(f"\n✅ 테스트 정확도: {test_acc*100:.2f}%")

cm = confusion_matrix(y_test, y_pred)
print(f"\n📊 Confusion Matrix:")
print(f"실제 go:   {cm[0][0]:4d}(정답)  {cm[0][1]:4d}(오답)")
print(f"실제 stop: {cm[1][0]:4d}(오답)  {cm[1][1]:4d}(정답)")

categories = ['go', 'stop']
report = classification_report(y_test, y_pred, target_names=categories, digits=4)
print(f"\n📋 Classification Report:")
print(report)

# ==================== 6. 노이즈 영향도 분석 ====================
print("\n" + "="*60)
print("🔍 노이즈 영향도 분석")
print("="*60)

for i, m in enumerate(meta_test):
    m['prediction'] = int(y_pred[i])
    m['true_label'] = int(y_test[i])
    m['correct'] = (y_pred[i] == y_test[i])
    m['confidence'] = float(y_pred_proba[i][m['prediction']])

def analyze_noise_impact(meta_test):
    results = []
    for m in meta_test:
        results.append({
            'distance_score': m['camera_params']['distance']['score'],
            'illumination_score': m['camera_params']['illumination']['score'],
            'background_score': m['camera_params']['background']['score'],
            'distance_type': m['camera_params']['distance']['type'],
            'illumination_type': m['camera_params']['illumination']['type'],
            'background_type': m['camera_params']['background']['type'],
            'correct': m['correct'],
            'confidence': m['confidence'],
            'category': m['category'],
            'font_type': m['font_type']
        })
    return pd.DataFrame(results)

df_results = analyze_noise_impact(meta_test)

csv_path = os.path.join(RESULTS_DIR, "detailed_results.csv")
df_results.to_csv(csv_path, index=False, encoding='utf-8-sig')
print(f"💾 저장: {csv_path}")

print(f"\n📊 전체 정확도: {df_results['correct'].mean()*100:.2f}%")

print("\n📏 거리별 정확도:")
for score in sorted(df_results['distance_score'].unique()):
    subset = df_results[df_results['distance_score'] == score]
    acc = subset['correct'].mean()
    dist_type = subset['distance_type'].iloc[0]
    print(f"  {score}. {dist_type:15s}: {acc*100:5.2f}% ({len(subset)}개)")

print("\n💡 조도별 정확도:")
for score in sorted(df_results['illumination_score'].unique()):
    subset = df_results[df_results['illumination_score'] == score]
    acc = subset['correct'].mean()
    illum_type = subset['illumination_type'].iloc[0]
    print(f"  {score}. {illum_type:15s}: {acc*100:5.2f}% ({len(subset)}개)")

print("\n🖼️  배경별 정확도:")
for score in sorted(df_results['background_score'].unique()):
    subset = df_results[df_results['background_score'] == score]
    acc = subset['correct'].mean()
    bg_type = subset['background_type'].iloc[0]
    print(f"  {score}. {bg_type:15s}: {acc*100:5.2f}% ({len(subset)}개)")

print("\n🔤 폰트 타입별 정확도:")
for font_type in sorted(df_results['font_type'].unique()):
    subset = df_results[df_results['font_type'] == font_type]
    acc = subset['correct'].mean()
    print(f"  {font_type:15s}: {acc*100:5.2f}% ({len(subset)}개)")

# ==================== 7. 3D 시각화 ====================
print("\n📊 3D 시각화 생성 중...")

fig = plt.figure(figsize=(20, 12))

# 3D Scatter
ax1 = fig.add_subplot(2, 3, 1, projection='3d')
correct_data = df_results[df_results['correct'] == True]
incorrect_data = df_results[df_results['correct'] == False]

ax1.scatter(correct_data['distance_score'], 
           correct_data['illumination_score'], 
           correct_data['background_score'],
           c='green', marker='o', s=30, alpha=0.5, label='정확')

ax1.scatter(incorrect_data['distance_score'], 
           incorrect_data['illumination_score'], 
           incorrect_data['background_score'],
           c='red', marker='x', s=80, alpha=0.8, label='오류')

ax1.set_xlabel('거리 레벨')
ax1.set_ylabel('조도 레벨')
ax1.set_zlabel('배경 레벨')
ax1.set_title('3D 노이즈 공간: 인식 vs 오인식', fontsize=12, fontweight='bold')
ax1.legend()

# 히트맵들
ax2 = fig.add_subplot(2, 3, 2)
pivot = df_results.groupby(['distance_score', 'illumination_score'])['correct'].mean().unstack()
sns.heatmap(pivot, annot=True, fmt='.2f', cmap='RdYlGn', vmin=0, vmax=1, ax=ax2)
ax2.set_title('거리 × 조도 정확도')

ax3 = fig.add_subplot(2, 3, 3)
pivot = df_results.groupby(['distance_score', 'background_score'])['correct'].mean().unstack()
sns.heatmap(pivot, annot=True, fmt='.2f', cmap='RdYlGn', vmin=0, vmax=1, ax=ax3)
ax3.set_title('거리 × 배경 정확도')

ax4 = fig.add_subplot(2, 3, 4)
pivot = df_results.groupby(['illumination_score', 'background_score'])['correct'].mean().unstack()
sns.heatmap(pivot, annot=True, fmt='.2f', cmap='RdYlGn', vmin=0, vmax=1, ax=ax4)
ax4.set_title('조도 × 배경 정확도')

ax5 = fig.add_subplot(2, 3, 5)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax5, 
            xticklabels=['go', 'stop'], yticklabels=['go', 'stop'])
ax5.set_title('Confusion Matrix')

ax6 = fig.add_subplot(2, 3, 6)
ax6.axis('off')
summary = f"""
📊 분석 결과

정확도: {test_acc*100:.1f}%
테스트: {len(y_test)}개
정답: {df_results['correct'].sum()}
오답: {(~df_results['correct']).sum()}

폰트별:
• Normal: {df_results[df_results['font_type']=='normal']['correct'].mean()*100:.1f}%
• Distorted: {df_results[df_results['font_type']=='distorted']['correct'].mean()*100:.1f}%
"""
ax6.text(0.1, 0.5, summary, fontsize=11, family='monospace', 
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.3))

plt.tight_layout()
viz_path = os.path.join(RESULTS_DIR, "noise_analysis_3d.png")
plt.savefig(viz_path, dpi=150)
print(f"✅ 저장: {viz_path}")
plt.close()

# ==================== 8. 오류율 분석 ====================
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 거리별 오류율
ax = axes[0, 0]
error_by_dist = df_results.groupby('distance_score').agg({
    'correct': lambda x: (1 - x.mean()) * 100
}).reset_index()
bars = ax.bar(error_by_dist['distance_score'], error_by_dist['correct'], 
              color=['#2ecc71', '#3498db', '#f39c12', '#e74c3c', '#c0392b'],
              edgecolor='black', linewidth=1.5)
ax.set_xlabel('거리 레벨')
ax.set_ylabel('오류율 (%)')
ax.set_title('거리별 오류율')
ax.grid(axis='y', alpha=0.3)
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.1f}%', ha='center', va='bottom', fontweight='bold')

# 조도별 오류율
ax = axes[0, 1]
error_by_illum = df_results.groupby('illumination_score').agg({
    'correct': lambda x: (1 - x.mean()) * 100
}).reset_index()
bars = ax.bar(error_by_illum['illumination_score'], error_by_illum['correct'],
              color=['#9b59b6', '#3498db', '#2ecc71', '#f39c12', '#e74c3c'],
              edgecolor='black', linewidth=1.5)
ax.set_xlabel('조도 레벨')
ax.set_ylabel('오류율 (%)')
ax.set_title('조도별 오류율')
ax.grid(axis='y', alpha=0.3)
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.1f}%', ha='center', va='bottom', fontweight='bold')

# 배경별 오류율
ax = axes[1, 0]
error_by_bg = df_results.groupby('background_score').agg({
    'correct': lambda x: (1 - x.mean()) * 100
}).reset_index()
bars = ax.bar(error_by_bg['background_score'], error_by_bg['correct'],
              color=['#1abc9c', '#3498db', '#f39c12', '#e67e22', '#e74c3c'],
              edgecolor='black', linewidth=1.5)
ax.set_xlabel('배경 레벨')
ax.set_ylabel('오류율 (%)')
ax.set_title('배경별 오류율')
ax.grid(axis='y', alpha=0.3)
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.1f}%', ha='center', va='bottom', fontweight='bold')

# 폰트 타입별 오류율
ax = axes[1, 1]
error_by_font = df_results.groupby('font_type').agg({
    'correct': lambda x: (1 - x.mean()) * 100
}).reset_index()
bars = ax.bar(error_by_font['font_type'], error_by_font['correct'],
              color=['#3498db', '#e74c3c'],
              edgecolor='black', linewidth=1.5)
ax.set_xlabel('폰트 타입')
ax.set_ylabel('오류율 (%)')
ax.set_title('폰트 타입별 오류율')
ax.grid(axis='y', alpha=0.3)
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.1f}%', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
viz_path_2 = os.path.join(RESULTS_DIR, "error_rate_analysis.png")
plt.savefig(viz_path_2, dpi=150)
print(f"✅ 저장: {viz_path_2}")
plt.close()

print("\n✨ 분석 완료!")
print(f"📁 결과: {RESULTS_DIR}")
print(f"📊 정확도: {test_acc*100:.2f}%")


🤖 PiCar-X 로봇 카메라 이미지 분류 및 노이즈 영향도 분석

📁 데이터셋 폴더 경로를 입력해주세요.

🔍 현재 디렉토리에서 발견된 데이터셋 폴더:
   1. robot_camera_dataset_20260218_204610
   2. robot_camera_dataset_20260218_205008
   3. robot_camera_dataset_20260218_205132
   4. robot_camera_dataset_20260218_205745



📌 번호를 입력하거나 전체 경로를 입력하세요:  4



✅ 선택된 폴더: D:\work\robot_camera_dataset_20260218_205745
📊 결과 저장: D:\work\robot_camera_dataset_20260218_205745\analysis_results

📂 메타데이터 로딩...
✅ 2000개 로드

🖼️  이미지 데이터 로딩...
  진행: 500/2000
  진행: 1000/2000
  진행: 1500/2000
  진행: 2000/2000

✅ 총 2000개 이미지 로드
  - shape: (2000, 8192)
  - go=1000개, stop=1000개

📊 데이터 분할:
  - 학습: 1600개
  - 테스트: 400개

🧠 신경망 모델 학습
✅ 학습 완료 (소요: 20.71초)
📊 반복: 30회
💾 모델 저장: D:\work\robot_camera_dataset_20260218_205745\analysis_results\robot_classifier_model.pkl

📈 모델 평가

✅ 테스트 정확도: 92.00%

📊 Confusion Matrix:
실제 go:    177(정답)    23(오답)
실제 stop:    9(오답)   191(정답)

📋 Classification Report:
              precision    recall  f1-score   support

          go     0.9516    0.8850    0.9171       200
        stop     0.8925    0.9550    0.9227       200

    accuracy                         0.9200       400
   macro avg     0.9221    0.9200    0.9199       400
weighted avg     0.9221    0.9200    0.9199       400


🔍 노이즈 영향도 분석
💾 저장: D:\work\robot_camera_dataset_20260218_2